In [1]:
import torch
import numpy as np

In [2]:
from load_dataset import LoadDataset
from model import MLP
from torch.utils.data import DataLoader

In [3]:
from training import Trainer
from metrics import Evaluator

In [4]:
from gradients import GradMonitor 
import matplotlib.pyplot as plt  

In [5]:
iris_path = '/home/nascj/knowledge_distillation/dataset/Iris.csv'
iris = LoadDataset(iris_path)
iris.load()
train_data, test_data = iris.split_dataset()

In [6]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [7]:
config_iris_MLP = {
    'input_dim': 4,
    'in_channels': 1,   
    'output_dim': 3,     # RGB
    'fc': [5],
    'criterion':  torch.nn.CrossEntropyLoss(),
    'optimizer':  torch.optim.Adam,
    'lr': 0.001
}

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
student = MLP(config_iris_MLP).to(device)
teacher = MLP(config_iris_MLP).to(device)
trainer = Trainer(teacher, config_iris_MLP)
epochs = 1000
ptrain_old = Evaluator(teacher).accuracy(train_loader)
patience = 0
for _ in range(epochs):
    trainer.train_epoch(train_loader)
    ptrain = Evaluator(teacher).accuracy(train_loader)
    if abs(ptrain_old - ptrain) < 1e-3:  # Convergence check
        patience += 1
    else:
        patience = 0
    ptrain_old = ptrain
    if patience >= 60:  # Early stopping after 5 epochs without improvement
        print(f"Early stopping at epoch {_+1}")
        break
metrics = Evaluator(teacher)
print(metrics.accuracy(train_loader), metrics.accuracy(test_loader))
grad_teacher = GradMonitor(teacher, config_iris_MLP)
info_teacher = grad_teacher.get_gradients(train_loader)


Early stopping at epoch 394
0.9523809523809523 1.0


In [ ]:
def get_student_model(T, c, teacher, train_loader, config):
   student = MLP(config).to(device)
   t_student = Trainer(student, config)
   sepochs = 1000
   ptrain = Evaluator(teacher).accuracy(train_loader)
   for k in range(sepochs):
       t_student.train_student(teacher, train_loader, T=T, c=c)
       strain = Evaluator(student).accuracy(train_loader)
       if abs(strain - ptrain) < 0.001:  # Convergence check
           print(k)
           break
   return student, k

In [ ]:
estudantes = []
prof = teacher
test_prec = [Evaluator(teacher).accuracy(test_loader)]
for t,c in zip([1,2,5,10],[0,0,0,0]):
   print(t,c)
   for i in range(30):
      student, k = get_student_model(T=t, c=c, teacher=prof, train_loader=train_loader)
      estudantes.append(student)
      test_prec.append(Evaluator(student).accuracy(test_loader))
      print(f"Student {i+1} trained in {k} epochs with test accuracy: {test_prec[-1]:.4f}")
      prof = student  # ✓ Student i+1 learns from Student i

362
Student 1 trained in 362 epochs with test accuracy: 1.0000
301
Student 2 trained in 301 epochs with test accuracy: 1.0000
577
Student 3 trained in 577 epochs with test accuracy: 1.0000
409
Student 4 trained in 409 epochs with test accuracy: 1.0000
384
Student 5 trained in 384 epochs with test accuracy: 1.0000
357
Student 6 trained in 357 epochs with test accuracy: 1.0000
308
Student 7 trained in 308 epochs with test accuracy: 1.0000
319
Student 8 trained in 319 epochs with test accuracy: 1.0000
572
Student 9 trained in 572 epochs with test accuracy: 1.0000
525
Student 10 trained in 525 epochs with test accuracy: 1.0000
331
Student 11 trained in 331 epochs with test accuracy: 1.0000
306
Student 12 trained in 306 epochs with test accuracy: 1.0000
400
Student 13 trained in 400 epochs with test accuracy: 1.0000
258
Student 14 trained in 258 epochs with test accuracy: 0.9778
255
Student 15 trained in 255 epochs with test accuracy: 1.0000
229
Student 16 trained in 229 epochs with test ac

KeyboardInterrupt: 

In [ ]:
plt.plot(list(range(len(test_prec))), test_prec)


##### Será que as informações dentro as redes sao iguais? -> vou dar uma olhada nos gradientes
---

In [10]:
teacher_grad_all = {}
student_grad_all = {}
output_student_all = {}
output_teacher_all = {}
pred_student_all = {}
pred_teacher_all = {}
for params in st_models:
    grad_student = GradMonitor(st_models[params], config_iris_MLP)
    info_student = grad_student.get_gradients(train_loader) 
    teacher_grad = []
    student_grad = []
    output_teacher = []
    output_student = []
    pred_teacher = []
    pred_student = []
    for i in range(len(info_student)):
        teacher_grad.append(info_teacher[i]['grads']['layers.fc1.weight'])
        student_grad.append(info_student[i]['grads']['layers.fc1.weight'])
        output_student.append(info_student[i]['grads']['layers.fc2.weight'])
        output_teacher.append(info_teacher[i]['grads']['layers.fc2.weight'])
        pred_student.append(info_student[i]['pred_labels'])
        pred_teacher.append(info_teacher[i]['pred_labels'])
    teacher_grad = torch.cat(teacher_grad, dim=0)
    student_grad = torch.cat(student_grad, dim=0)
    output_teacher = torch.cat(output_teacher, dim=0)
    output_student = torch.cat(output_student, dim=0)
    pred_teacher = torch.cat(pred_teacher, dim=0)
    pred_student = torch.cat(pred_student, dim=0)  
    torder = np.argsort(pred_teacher.cpu().numpy())
    sorder = np.argsort(pred_student.cpu().numpy())
    teacher_grad_sorted = teacher_grad[torder]
    student_grad_sorted = student_grad[sorder]
    output_student_sorted = output_student[sorder]
    output_teacher_sorted = output_teacher[torder] 
    pred_student_all[params] = pred_student
    pred_teacher_all[params] = pred_teacher
    teacher_grad_all[params] = teacher_grad_sorted
    student_grad_all[params] = student_grad_sorted
    output_student_all[params] = output_student_sorted
    output_teacher_all[params] = output_teacher_sorted  

In [11]:
from pruning_utils import compute_similarity_matrix

In [12]:
all_teacher_similarity = {}
all_student_similarity = {}
all_output_teacher_similarity = {}
all_output_student_similarity = {}
for params in student_grad_all:
    teacher_similarity = compute_similarity_matrix(teacher_grad_all[params])
    student_similarity = compute_similarity_matrix(student_grad_all[params])
    output_teacher_similarity = compute_similarity_matrix(output_teacher_all[params])
    output_student_similarity = compute_similarity_matrix(output_student_all[params])
    all_teacher_similarity[params] = teacher_similarity
    all_student_similarity[params] = student_similarity
    all_output_teacher_similarity[params] = output_teacher_similarity
    all_output_student_similarity[params] = output_student_similarity